# 🏧 ATM System – Object Oriented Programming in Python
**Project by Nishika Gupta**

This project simulates a real-world ATM system using core OOP principles:
- **Encapsulation** – account data is protected using private attributes
- **Abstraction** – users interact via clean methods without knowing internals
- **Classes & Objects** – `BankAccount` and `ATM` classes model real entities
- **Exception Handling** – invalid operations are caught gracefully


In [ ]:
# ============================================================
# ATM SYSTEM — Object Oriented Python
# ============================================================

from datetime import datetime

class BankAccount:
    """
    Represents a bank account with encapsulated balance and transaction history.
    Demonstrates: Encapsulation, Data Protection, OOP Design
    """

    def __init__(self, account_number, holder_name, pin, initial_balance=0.0):
        self.account_number = account_number
        self.holder_name = holder_name
        self.__pin = pin                          # private attribute
        self.__balance = initial_balance          # private attribute
        self.__transactions = []                  # private transaction log

    def verify_pin(self, pin):
        """Verify user PIN securely."""
        return self.__pin == pin

    def get_balance(self):
        """Return current balance."""
        return self.__balance

    def deposit(self, amount):
        """Deposit money into the account."""
        if amount <= 0:
            raise ValueError("Deposit amount must be greater than zero.")
        self.__balance += amount
        self.__log_transaction("DEPOSIT", amount)
        return self.__balance

    def withdraw(self, amount):
        """Withdraw money from the account."""
        if amount <= 0:
            raise ValueError("Withdrawal amount must be greater than zero.")
        if amount > self.__balance:
            raise ValueError("Insufficient funds.")
        self.__balance -= amount
        self.__log_transaction("WITHDRAWAL", amount)
        return self.__balance

    def get_transaction_history(self):
        """Return a copy of transaction history."""
        return list(self.__transactions)

    def __log_transaction(self, txn_type, amount):
        """Private method to log transactions with timestamp."""
        self.__transactions.append({
            "type": txn_type,
            "amount": amount,
            "balance_after": self.__balance,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

    def __str__(self):
        return f"Account [{self.account_number}] | Holder: {self.holder_name}"


In [ ]:
class ATM:
    """
    ATM machine that manages multiple accounts and handles user sessions.
    Demonstrates: Abstraction, Composition, Multiple Object Management
    """

    MAX_PIN_ATTEMPTS = 3

    def __init__(self, atm_id, location):
        self.atm_id = atm_id
        self.location = location
        self.__accounts = {}           # stores all accounts
        self.__current_account = None  # active session

    def add_account(self, account: BankAccount):
        """Register a bank account with this ATM."""
        self.__accounts[account.account_number] = account
        print(f"✅ Account {account.account_number} registered successfully.")

    def login(self, account_number, pin):
        """Authenticate user and start a session."""
        if account_number not in self.__accounts:
            print("❌ Account not found.")
            return False

        account = self.__accounts[account_number]
        for attempt in range(1, self.MAX_PIN_ATTEMPTS + 1):
            if account.verify_pin(pin):
                self.__current_account = account
                print(f"\n🔓 Welcome, {account.holder_name}!")
                return True
            else:
                remaining = self.MAX_PIN_ATTEMPTS - attempt
                print(f"❌ Incorrect PIN. {remaining} attempt(s) remaining.")
        print("🔒 Account locked due to too many failed attempts.")
        return False

    def logout(self):
        """End the current session."""
        if self.__current_account:
            print(f"\n👋 Goodbye, {self.__current_account.holder_name}! Session ended.")
            self.__current_account = None

    def check_balance(self):
        """Display balance of logged-in account."""
        self.__require_login()
        bal = self.__current_account.get_balance()
        print(f"\n💰 Current Balance: ₹{bal:,.2f}")

    def deposit(self, amount):
        """Deposit money into logged-in account."""
        self.__require_login()
        try:
            new_bal = self.__current_account.deposit(amount)
            print(f"✅ ₹{amount:,.2f} deposited. New Balance: ₹{new_bal:,.2f}")
        except ValueError as e:
            print(f"❌ Error: {e}")

    def withdraw(self, amount):
        """Withdraw money from logged-in account."""
        self.__require_login()
        try:
            new_bal = self.__current_account.withdraw(amount)
            print(f"✅ ₹{amount:,.2f} withdrawn. New Balance: ₹{new_bal:,.2f}")
        except ValueError as e:
            print(f"❌ Error: {e}")

    def print_statement(self):
        """Print full transaction history."""
        self.__require_login()
        history = self.__current_account.get_transaction_history()
        print(f"\n📄 Transaction Statement for {self.__current_account.holder_name}")
        print("-" * 60)
        if not history:
            print("No transactions yet.")
        for txn in history:
            symbol = "⬆️" if txn['type'] == "DEPOSIT" else "⬇️"
            print(f"{symbol} {txn['type']:<12} ₹{txn['amount']:>10,.2f}   "
                  f"Balance: ₹{txn['balance_after']:>10,.2f}   {txn['timestamp']}")
        print("-" * 60)

    def __require_login(self):
        """Raise error if no active session."""
        if not self.__current_account:
            raise PermissionError("No active session. Please login first.")

    def __str__(self):
        return f"ATM [{self.atm_id}] at {self.location}"


## 🧪 Demo – Running the ATM System

In [ ]:
# --- Setup ATM and Accounts ---
atm = ATM(atm_id="ATM001", location="Thapar Institute, Patiala")

acc1 = BankAccount(account_number="ACC1001", holder_name="Nishika Gupta",   pin="2408", initial_balance=10000)
acc2 = BankAccount(account_number="ACC1002", holder_name="Rahul Sharma",    pin="1234", initial_balance=5000)

atm.add_account(acc1)
atm.add_account(acc2)

In [ ]:
# --- Login and perform transactions ---
print("=" * 60)
print("         WELCOME TO NISHIKA'S ATM SYSTEM")
print("=" * 60)

atm.login("ACC1001", "2408")
atm.check_balance()
atm.deposit(5000)
atm.withdraw(2000)
atm.withdraw(50000)   # should fail – insufficient funds
atm.deposit(-100)     # should fail – invalid amount
atm.print_statement()
atm.logout()

In [ ]:
# --- Test wrong PIN / security ---
print("\n" + "=" * 60)
print("         TESTING SECURITY FEATURES")
print("=" * 60)
atm.login("ACC1002", "9999")  # wrong PIN – 3 attempts used

## 📌 OOP Concepts Demonstrated

| Concept | Where Used |
|---|---|
| **Encapsulation** | `__balance`, `__pin`, `__transactions` are private |
| **Abstraction** | Users call `deposit()`, `withdraw()` without knowing internals |
| **Classes & Objects** | `BankAccount` and `ATM` model real-world entities |
| **Composition** | `ATM` contains multiple `BankAccount` objects |
| **Exception Handling** | Invalid amounts and insufficient funds handled gracefully |
| **Private Methods** | `__log_transaction()`, `__require_login()` |
